In [64]:
import pandas as pd
pd.set_option("display.max_rows", 10)

import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.common.keys import Keys
from IPython.display import Image

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--headless=new") # 'new' is required for Selenium 4+
driver = webdriver.Chrome(options=chrome_options)

wait = WebDriverWait(driver, 10)

driver.get("https://webscraper.io/test-sites/pagination")

# driver.set_window_size(1100, 1200)

In [52]:
# Fields on the listing pages: Title, Tagline, Year, Country of origin, Mileage, Price, Rarity Rating, Status, Product URL, Thumbnail URL
def page_scarper(driver) -> list:
    cars = []
    car_cards = driver.find_elements(By.CSS_SELECTOR, "div.card.sitemap-card.test-sites-card")

    for car in car_cards:
        title = car.find_element(By.CLASS_NAME, "card-title").text.strip()
        
        tagline = car.find_element(By.CLASS_NAME, "description").text.strip()
        
        body_style = driver.find_element(By.CSS_SELECTOR,"a.nav-link.subcategory-link.active").text.strip()
        
        card_text = car.find_elements(By.CLASS_NAME, "card-text")
        
        year = card_text[0].text.strip().replace("Year: ",  "")
        
        country_of_origin = card_text[1].text.strip().replace("Country of origin: ",  "")
        
        mileage = card_text[2].text.replace("Mileage:",  "").replace(" ","").replace("km", "")
        
        price = car.find_element(By.CLASS_NAME, "price").text.replace("USD","").replace(" ","")
        rarity_rating = car.find_element(By.CLASS_NAME, "col-6").get_attribute("data-rating")
        
        status_and_count = car.find_element(By.CLASS_NAME, "badge").text.replace(" ","")
        if ":" in status_and_count:
            status =  status_and_count.split(sep=":")[0]
            count = status_and_count.split(sep=":")[1]
        else:
            status = status_and_count
            count = 0
        cars.append({
            "title": title,
            "tagline": tagline,
            "body_style": body_style,
            "year": year,
            "country_of_origin": country_of_origin,
            "mileage": mileage,
            "price": price,
            "rarity_rating": rarity_rating,
            "status": status,
            "count": count,
        })
    print("Scraped:", len(cars),"Cars")
    return cars

In [59]:
selector = ".category-link.nav-link"
count = len(driver.find_elements(By.CSS_SELECTOR, selector))
scraped_cars = []

for i in range(count):
    # Re-find elements after each click, since the page may have changed.
    nav_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
    category = WebDriverWait(driver, 10).until(EC.element_to_be_clickable(nav_buttons[i]))

    print("== Scraping Catagory:", category.text)

    category.click()
    
    active_nav = driver.find_element(By.CSS_SELECTOR, ".nav-item.active")
    count_sub = active_nav.find_elements(By.CSS_SELECTOR, ".nav-link.subcategory-link")
    
    for i in range(len(count_sub)):
        active_nav = driver.find_element(By.CSS_SELECTOR, ".nav-item.active")
        sub_button = active_nav.find_elements(By.CSS_SELECTOR, ".nav-link.subcategory-link")
        
        print("- Scraping Subcatagory:",sub_button[i].text)
        
        subcategory = WebDriverWait(driver, 15).until(EC.element_to_be_clickable(sub_button[i]))
        subcategory.click()
        time.sleep(1) 
        scraped_cars.extend(page_scarper(driver))

        page_number = 1
        
        while True:
            try:
                next_link = driver.find_element(By.CSS_SELECTOR, "a.page-link.next")
                next_link.click()
                page_number += 1
                print("-> Moving to page:", page_number)
            except:    
                print("Scraped One Page")
                break
            time.sleep(1)
            scraped_cars.extend(page_scarper(driver))
            
# driver.quit()

== Scraping Catagory: BMW
- Scraping Subcatagory: Convertible
Scraped: 6 Cars
-> Moving to page: 2
Scraped: 2 Cars
Scraped One Page
- Scraping Subcatagory: Coupe
Scraped: 1 Cars
Scraped One Page
- Scraping Subcatagory: Hatchback
Scraped: 4 Cars
Scraped One Page
- Scraping Subcatagory: Sedan
Scraped: 3 Cars
Scraped One Page
== Scraping Catagory: Ferrari
- Scraping Subcatagory: Convertible
Scraped: 6 Cars
Scraped One Page
- Scraping Subcatagory: Coupe
Scraped: 5 Cars
Scraped One Page
- Scraping Subcatagory: Hatchback
Scraped: 1 Cars
Scraped One Page
- Scraping Subcatagory: Sedan
Scraped: 1 Cars
Scraped One Page
== Scraping Catagory: Ford
- Scraping Subcatagory: Convertible
Scraped: 4 Cars
Scraped One Page
- Scraping Subcatagory: Coupe
Scraped: 6 Cars
Scraped One Page
- Scraping Subcatagory: Sedan
Scraped: 6 Cars
-> Moving to page: 2
Scraped: 1 Cars
Scraped One Page
== Scraping Catagory: Jaguar
- Scraping Subcatagory: Convertible
Scraped: 6 Cars
Scraped One Page
- Scraping Subcatagory: Co

In [ ]:
spacer ="\n-----------------------------------\n"
print("Listing Scraped Data Into a Table...")
cars = pd.DataFrame(scraped_cars)
time.sleep(2)
print("Listed!")
time.sleep(1)
print("Table Data Types Before Cleaning", spacer, cars.dtypes)

print("Fixing Data Types For Columns...\n")
time.sleep(2)
to_numeric = ["year", "mileage", "rarity_rating", "count"]
for col in to_numeric:
    cars[col] = pd.to_numeric(cars[col], errors="coerce")

print("Fixing Data Types For The Price Column...", spacer)
time.sleep(2)
cars["price"] = cars["price"].astype(float).copy()
print("", cars.dtypes)


cars["brand"] = cars["title"].map(lambda x: x.split()[0])

cars['title'] = cars['title'].map(lambda x: x[:-4] )

print("Fix Done!\n")
display(cars.head(1))

Listing Scraped Data Into a Table...
Listed!
Table Data Types Before Cleaning 
-----------------------------------
 title                   str
tagline                 str
body_style              str
year                    str
country_of_origin       str
mileage                 str
price                   str
rarity_rating           str
status                  str
count                object
dtype: object
Fixing Data Types For Columns...

Fixing Data Types For The Price Column... 
-----------------------------------

 title                    str
tagline                  str
body_style               str
year                   int64
country_of_origin        str
mileage                int64
price                float64
rarity_rating          int64
status                   str
count                  int64
dtype: object
Fix Done!



,title,tagline,body_style,year,country_of_origin,mileage,price,rarity_rating,status,count,brand
0,BMW 2002,"Concours 2002, five-speed",Convertible,1978,Italy,181589,336315.0,5,Reserved,0,BMW


In [49]:
driver.quit()

driver = webdriver.Chrome()

driver.get("https://webscraper.io/test-sites/pagination")

'''if category.text == "Jaguar":
    announcement = driver.find_element(By.CSS_SELECTOR, ".v2-announcement-try")
    wait.until(EC.invisibility_of_element_located(announcement))'''

#page_scarper(driver)
"""https://webscraper.io/test-sites/pagination?page=17"""

'https://webscraper.io/test-sites/pagination?page=17'